# 🧬 Pavan Krovvidi FingerWave Neural Network: test it yourself

*"Every life has a unique print. So does every prediction."*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Pavankrovvidi/pavan-krovvidi-fingerwave/blob/main/examples/FingerWave_Quickstart.ipynb)
[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.23766123.svg)](https://doi.org/10.5281/zenodo.23766123)

FingerWave is a neural network inspired by how fingerprints form in the womb. It repairs damaged structural data by
**starting from what it can trust**, **spreading information along the structure like a wave**, and **never inventing
or losing the special points (cores and deltas)** that define the pattern.

This notebook runs on a free Colab CPU in about two minutes. You will:

1. install the package from PyPI,
2. repair damaged fingerprints with the pretrained model,
3. **damage a fingerprint yourself** and watch FingerWave repair it,
4. measure the results on 100 new fingerprints.

Created by **Pavan Kumar Krovvidi** · [GitHub](https://github.com/Pavankrovvidi/pavan-krovvidi-fingerwave) · [LinkedIn](https://www.linkedin.com/in/pavan-kumar-krovvidi-927290250) · Apache 2.0

## 1. Install

In [ ]:
!pip install -q "pavan-krovvidi-fingerwave[viz]"

## 2. Meet the algorithm

In [ ]:
import torch, matplotlib.pyplot as plt
import pavan_krovvidi_fingerwave as pk
from pavan_krovvidi_fingerwave import PavanKrovvidiFingerWaveNetwork
from pavan_krovvidi_fingerwave.data import make_dataset, make_model_input
from pavan_krovvidi_fingerwave.metrics import evaluate
from pavan_krovvidi_fingerwave.topo_loss import count_defects
from pavan_krovvidi_fingerwave.viz import plot_field

pk.about()
model = PavanKrovvidiFingerWaveNetwork.from_pretrained()
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 3. Repair damaged fingerprints

The generator creates fingerprints the model has never seen, damages them (noise, blur, smudges, missing bands) and keeps
the exact ground truth. Colour shows ridge direction; ○ marks cores and △ marks deltas.

In [ ]:
data = make_dataset(4, seed=123)          # 4 new fingerprints with exact ground truth
x = make_model_input(data)                # (4, 4, 64, 64): image, orientation u, v, coherence
with torch.no_grad():
    pred = model(x)                       # (4, 2, 64, 64): repaired doubled-angle field

fig, ax = plt.subplots(3, 4, figsize=(12, 9.5))
for i in range(4):
    plot_field(data["w_obs"][i:i+1], ax[0, i], "damaged input")
    plot_field(pred[i:i+1],          ax[1, i], "FingerWave")
    plot_field(data["w_true"][i:i+1], ax[2, i], "ground truth")
fig.suptitle("Pavan Krovvidi FingerWave Neural Network", fontsize=14)
plt.tight_layout(); plt.show()

## 4. 🔨 Break it yourself

Cut a hole into a fingerprint: inside the hole the model gets **no information at all**. Change the numbers and run the
cell again. Try a bigger hole, move it onto the core, or try another fingerprint with `SAMPLE`.

In [ ]:
SAMPLE    = 0     # which fingerprint (0-3)
HOLE_X    = 24    # left edge of the hole (0-63)
HOLE_Y    = 16    # top edge of the hole (0-63)
HOLE_SIZE = 20    # width and height of the hole in grid cells

xb = x[SAMPLE:SAMPLE+1].clone()
xb[:, :, HOLE_Y:HOLE_Y+HOLE_SIZE, HOLE_X:HOLE_X+HOLE_SIZE] = 0.0     # erase image, orientation and coherence
with torch.no_grad():
    repaired = model(xb)

truth = data["w_true"][SAMPLE:SAMPLE+1]
damaged = xb[:, 1:3]
fig, ax = plt.subplots(1, 3, figsize=(12, 4.4))
plot_field(damaged, ax[0], f"your damage: {HOLE_SIZE}x{HOLE_SIZE} hole")
plot_field(repaired, ax[1], "FingerWave repair")
plot_field(truth, ax[2], "ground truth")
for a in ax[:2]:
    a.add_patch(plt.Rectangle((HOLE_X - .5, HOLE_Y - .5), HOLE_SIZE, HOLE_SIZE, fill=False, ec="white", ls="--", lw=1.5))
plt.tight_layout(); plt.show()

got, _ = count_defects(repaired); want, _ = count_defects(truth)
same = (got["plus_half"], got["minus_half"]) == (want["plus_half"], want["minus_half"])
print(f"FingerWave: {got['plus_half']} cores, {got['minus_half']} deltas | truth: {want['plus_half']} cores, {want['minus_half']} deltas"
      f" -> {'topology recovered ✅' if same else 'topology changed ❌ (try a smaller hole)'}")

## 5. Measure it on 100 new fingerprints

`TER` is the Topological Error Rate: the share of prints with the wrong number of cores or deltas (lower is better).
`angle_occ` is the ridge-direction error inside hidden regions, in degrees.

In [ ]:
test = make_dataset(100, seed=7)
with torch.no_grad():
    out = model(make_model_input(test))
for k, v in evaluate(out, test).items():
    print(f"{k:>10}: {v:.3f}")

## What's next?

* ⭐ **Star the repo** if this was useful: [github.com/Pavankrovvidi/pavan-krovvidi-fingerwave](https://github.com/Pavankrovvidi/pavan-krovvidi-fingerwave)
* 🧪 **Try your own data**: any map with direction and gaps (fibres, flows, layers). See *Adapting FingerWave* in the README.
* 💬 **Tell me what you found**, good or bad, on [LinkedIn](https://www.linkedin.com/in/pavan-kumar-krovvidi-927290250) or in a GitHub issue.

**Cite:** Krovvidi, P. K. (2026). *Pavan Krovvidi FingerWave Neural Network.* Zenodo. https://doi.org/10.5281/zenodo.23766123